# NeuroGolf Solver Family: Fill / Additive Marking - nonlocal_1color

This notebook builds the first scoped submission for the `fill_enclosed_regions / nonlocal_1color` subtype.

Workflow:

1. Load the strict 41-task subtype from `task_type_map.csv`.
2. Try bounded wider additive-template models for tasks where 3x3 context is insufficient.
3. Fall back to compact additive candidates when they visibly fit.
4. Emit identity fallback models for any remaining task so the submission zip is complete.
5. Build `/kaggle/working/submission.zip` from exactly this subtype scope.

The generated maps are heuristic solver-routing labels. Validate against visible examples before submitting.

In [1]:
# Inline helper functions from submission_nbs/neurogolf_nb_common.py
"""Shared helpers for NeuroGolf submission notebooks.

The notebooks in this folder are solver-family workbooks. They should be
copied into Kaggle or run locally with the competition files available.
"""

import json
import math
import os
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np

try:
    import onnx
    import onnxruntime as ort
    from onnx import TensorProto, helper, numpy_helper
except Exception:  # Notebook analysis cells can still run without ONNX.
    onnx = None
    ort = None
    TensorProto = None
    helper = None
    numpy_helper = None




"""Shared helpers for NeuroGolf submission notebooks.

The notebooks in this folder are solver-family workbooks. They should be
copied into Kaggle or run locally with the competition files available.
"""


BATCH, CH, H, W = 1, 10, 30, 30


def default_paths():
    kaggle_dir = Path("/kaggle/input/competitions/neurogolf-2026")
    if kaggle_dir.exists():
        data_dir = kaggle_dir
        root = Path("/kaggle/working")
    else:
        root = Path.cwd()
        data_dir = root / "competition_material" / "taskfiles"
        if not data_dir.exists():
            data_dir = root / "competition_material"
    out_dir = root / "working_submission"
    out_dir.mkdir(parents=True, exist_ok=True)
    return data_dir, out_dir


def load_task_type_map(path="/kaggle/input/datasets/prince22466/task-type-map-csv/task_type_map.csv"):
    import pandas as pd

    candidates = [Path(path), Path("task_groups/task_type_map.csv")]
    for candidate in candidates:
        if candidate.exists():
            return pd.read_csv(candidate, dtype={"task_id": str})
    raise FileNotFoundError(f"task_type_map.csv not found in: {candidates}")


def load_task_groups(path="task_groups/task_type_groups.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def family_task_ids(family, groups_path="/kaggle/input/datasets/prince22466/task-type-groups-json/task_type_groups.json"):
    groups_candidates = [Path(groups_path), Path("task_groups/task_type_groups.json")]
    for candidate in groups_candidates:
        if candidate.exists():
            groups = load_task_groups(candidate)
            return groups.get(family, [])
    raise FileNotFoundError(f"task_type_groups.json not found in: {groups_candidates}")


def task_num(task_id):
    return int(str(task_id).replace("task", ""))


def task_path(data_dir, task_id):
    data_dir = Path(data_dir)
    name = f"{task_id}.json" if str(task_id).startswith("task") else f"task{int(task_id):03d}.json"
    direct = data_dir / name
    if direct.exists():
        return direct
    nested = data_dir / "taskfiles" / name
    if nested.exists():
        return nested
    raise FileNotFoundError(name)


def load_task(data_dir, task_id):
    with task_path(data_dir, task_id).open("r", encoding="utf-8") as f:
        return json.load(f)


def all_examples(task):
    return task.get("train", []) + task.get("test", []) + task.get("arc-gen", [])


def grid_shape(grid):
    return len(grid), len(grid[0]) if grid else 0


def grid_to_tensor(grid):
    arr = np.zeros((BATCH, CH, H, W), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if 0 <= r < H and 0 <= c < W:
                arr[0, int(color), r, c] = 1.0
    return arr


def tensor_to_grid(arr):
    arr = np.asarray(arr)
    if arr.ndim == 4:
        arr = arr[0]
    grid = []
    for r in range(H):
        row = []
        for c in range(W):
            vals = np.where(arr[:, r, c] > 0.5)[0]
            row.append(int(vals[0]) if len(vals) == 1 else 0)
        while row and row[-1] == 0:
            row.pop()
        grid.append(row)
    while grid and not grid[-1]:
        grid.pop()
    return grid


def require_onnx():
    if onnx is None or helper is None or TensorProto is None or numpy_helper is None:
        raise ImportError("onnx is required to build models")


def make_initializer(name, array):
    arr = np.asarray(array, dtype=np.float16)
    return numpy_helper.from_array(arr, name=name)


def make_model(nodes, initializers, opset=10):
    require_onnx()
    # Keep competition-facing graph I/O as float32. Cast internally to float16.
    inp = helper.make_tensor_value_info("input", TensorProto.FLOAT, [BATCH, CH, H, W])
    out = helper.make_tensor_value_info("output", TensorProto.FLOAT, [BATCH, CH, H, W])
    for node in nodes:
        for i, value in enumerate(node.input):
            if value == "input":
                node.input[i] = "input_f16"
        for i, value in enumerate(node.output):
            if value == "output":
                node.output[i] = "output_f16"
    cast_in = helper.make_node("Cast", ["input"], ["input_f16"], to=TensorProto.FLOAT16)
    cast_out = helper.make_node("Cast", ["output_f16"], ["output"], to=TensorProto.FLOAT)
    graph = helper.make_graph([cast_in] + list(nodes) + [cast_out], "graph", [inp], [out], initializers)
    return helper.make_model(graph, ir_version=10, opset_imports=[helper.make_opsetid("", opset)])


def make_identity_model():
    return make_1x1_color_model({c: c for c in range(CH)})


def make_1x1_color_model(mapping):
    require_onnx()
    dt = TensorProto.FLOAT16
    weights = np.zeros((CH, CH, 1, 1), dtype=np.float16)
    bias = np.full((CH,), -0.5, dtype=np.float16)
    for ic in range(CH):
        oc = int(mapping.get(ic, ic))
        weights[oc, ic, 0, 0] = 1.0
    w = make_initializer("W", weights)
    b = make_initializer("B", bias)
    node = helper.make_node("Conv", ["input", "W", "B"], ["output"], kernel_shape=[1, 1])
    return make_model([node], [w, b])


def infer_global_color_mapping(examples):
    mapping = {}
    for ex in examples:
        inp, out = ex["input"], ex["output"]
        if grid_shape(inp) != grid_shape(out):
            return None
        for r, row in enumerate(inp):
            for c, ic in enumerate(row):
                oc = out[r][c]
                prev = mapping.get(int(ic))
                if prev is None:
                    mapping[int(ic)] = int(oc)
                elif prev != int(oc):
                    return None
    for c in range(CH):
        mapping.setdefault(c, c)
    return mapping


def train_color_remap_model(task):
    mapping = infer_global_color_mapping(all_examples(task))
    if mapping is None:
        return None, {"ok": False, "reason": "no consistent global color mapping"}
    return make_1x1_color_model(mapping), {"ok": True, "mapping": mapping}


def fixed_transform(grid, transform):
    arr = np.array(grid, dtype=int)
    if transform == "rot90":
        return np.rot90(arr, -1).tolist()
    if transform == "rot180":
        return np.rot90(arr, 2).tolist()
    if transform == "rot270":
        return np.rot90(arr, 1).tolist()
    if transform == "flip_h":
        return np.fliplr(arr).tolist()
    if transform == "flip_v":
        return np.flipud(arr).tolist()
    if transform == "transpose":
        return arr.T.tolist()
    raise ValueError(transform)


def infer_fixed_geometric_transform(examples):
    names = ["rot90", "rot180", "rot270", "flip_h", "flip_v", "transpose"]
    matches = []
    for name in names:
        if all(fixed_transform(ex["input"], name) == ex["output"] for ex in examples):
            matches.append(name)
    return matches


def save_model(model, out_dir, task_id):
    require_onnx()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{task_id}.onnx"
    onnx.save(model, path)
    return path


def run_model(model_or_path, input_grid):
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required to run validation")
    if isinstance(model_or_path, (str, Path)):
        session = ort.InferenceSession(str(model_or_path), providers=["CPUExecutionProvider"])
    else:
        session = ort.InferenceSession(model_or_path.SerializeToString(), providers=["CPUExecutionProvider"])
    output = session.run(["output"], {"input": grid_to_tensor(input_grid)})[0]
    return (output > 0).astype(np.float32)


def visible_validation_summary(model_or_path, task, max_examples=None):
    examples = all_examples(task)
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong_expected = None
    first_wrong_actual =None
    print("len of examples: ", len(examples))

    count_ex =1
    for ex in examples:
        print(f"example {count_ex}")
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong_expected is None:
                first_wrong_expected = ex
                first_wrong_actual = actual

        count_ex = count_ex +1
        
    return {"right": right, "wrong": wrong, 
            "first_wrong_expected": first_wrong_expected, "first_wrong_actual":first_wrong_actual}


def split_examples(task):
    return {
        "train": task.get("train", []),
        "test": task.get("test", []),
        "arc_gen": task.get("arc-gen", []),
    }


def validation_summary_for_examples(model_or_path, examples, max_examples=None):
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    total = right + wrong
    accuracy = right / total if total else None
    return {"right": right, "wrong": wrong, "total": total, "accuracy": accuracy, "first_wrong": first_wrong}


def split_validation_summary(model_or_path, task, max_examples=None):
    rows = {}
    for split, examples in split_examples(task).items():
        summary = validation_summary_for_examples(model_or_path, examples, max_examples=max_examples)
        summary.pop("first_wrong", None)
        rows[split] = summary
    visible = validation_summary_for_examples(model_or_path, all_examples(task), max_examples=max_examples)
    visible.pop("first_wrong", None)
    rows["visible_all"] = visible
    return rows


def count_model_params(model_or_path):
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    params = 0
    for init in model.graph.initializer:
        if init.dims:
            params += math.prod(init.dims)
        else:
            params += 1
    for node in model.graph.node:
        if node.op_type != "Constant":
            continue
        for attr in node.attribute:
            if attr.name == "value":
                params += math.prod(attr.t.dims) if attr.t.dims else 1
            elif attr.name == "value_floats":
                params += len(attr.floats)
            elif attr.name == "value_ints":
                params += len(attr.ints)
            elif attr.name == "value_strings":
                params += len(attr.strings)
    return int(params)


def model_architecture_summary(model_or_path):
    require_onnx()
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    op_counts = Counter(node.op_type for node in model.graph.node)
    init_shapes = {init.name: list(init.dims) for init in model.graph.initializer}
    return {
        "ir_version": model.ir_version,
        "opsets": {op.domain or "ai.onnx": op.version for op in model.opset_import},
        "nodes": len(model.graph.node),
        "op_counts": dict(op_counts),
        "initializers": init_shapes,
        "params": count_model_params(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
    }


def approximate_memory_from_model_shapes(model_or_path):
    """Approximate scored tensor memory from static value_info shapes.

    The official helper uses ONNX Runtime profiling to refine tensor memory.
    This approximation is useful in notebooks before running the full profiler.
    """
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    inferred = onnx.shape_inference.infer_shapes(model)
    graph = inferred.graph
    total = 0
    for value in list(graph.value_info):
        tensor_type = value.type.tensor_type
        if not tensor_type.HasField("shape"):
            continue
        dims = []
        for dim in tensor_type.shape.dim:
            if not dim.HasField("dim_value") or dim.dim_value <= 0:
                dims = []
                break
            dims.append(dim.dim_value)
        if dims:
            total += math.prod(dims) * 2
    return int(total)


def runtime_memory_profile(model_or_path, sample_input_grid):
    """Return an approximate competition memory/param profile.

    This uses ONNX Runtime profiling if available. It is not a replacement for
    the official Kaggle validator, but it tracks the same concerns: parameters,
    intermediate tensor memory, and file size.
    """
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required for runtime memory profiling")
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    options = ort.SessionOptions()
    options.enable_profiling = True
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    session = ort.InferenceSession(model.SerializeToString(), options, providers=["CPUExecutionProvider"])
    session.run(["output"], {"input": grid_to_tensor(sample_input_grid)})
    trace_path = session.end_profiling()
    trace_memory = 0
    try:
        with open(trace_path, "r", encoding="utf-8") as f:
            trace = json.load(f)
        for event in trace:
            args = event.get("args", {})
            for shape_dict in args.get("output_type_shape", []) or []:
                for dims in shape_dict.values():
                    if dims and all(isinstance(d, int) and d > 0 for d in dims):
                        trace_memory += math.prod(dims) * 2
    except Exception:
        trace_memory = approximate_memory_from_model_shapes(model)
    return {
        "params": count_model_params(model),
        "runtime_memory_bytes": int(trace_memory),
        "static_memory_bytes": approximate_memory_from_model_shapes(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
        "profile_trace_path": trace_path,
    }


def model_report(model_or_path, task=None, sample_input_grid=None, max_validation_examples=None):
    report = {"architecture": model_architecture_summary(model_or_path)}
    if task is not None:
        report["performance"] = split_validation_summary(
            model_or_path,
            task,
            max_examples=max_validation_examples,
        )
        if sample_input_grid is None:
            examples = all_examples(task)
            if examples:
                sample_input_grid = examples[0]["input"]
    if sample_input_grid is not None and ort is not None:
        report["memory_profile"] = runtime_memory_profile(model_or_path, sample_input_grid)
    else:
        report["memory_profile"] = {
            "params": report["architecture"]["params"],
            "static_memory_bytes": approximate_memory_from_model_shapes(model_or_path),
            "runtime_memory_bytes": None,
            "file_size_bytes": report["architecture"]["file_size_bytes"],
            "profile_trace_path": None,
        }
    return report


def create_submission_zip(model_dir, zip_path=None):
    model_dir = Path(model_dir)
    if zip_path is None:
        zip_path = model_dir / "submission.zip"
    else:
        zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(model_dir.glob("task*.onnx")):
            zf.write(path, path.name)
    return zip_path


def build_family_submission(family, trainer, data_dir, out_dir, fallback_identity=False, validate=False, task_ids_override=None):
    task_ids = list(task_ids_override) if task_ids_override is not None else family_task_ids(family)
    rows = []
    for task_id in task_ids:
        task = load_task(data_dir, task_id)
        model, info = trainer(task)
        if model is None and fallback_identity:
            model = make_identity_model()
            info = {**info, "fallback": "identity"}
        if model is None:
            rows.append({"task_id": task_id, "saved": False, **info})
            continue
        path = save_model(model, out_dir, task_id)
        row = {"task_id": task_id, "saved": True, "path": str(path), **info}
        if validate:
            try:
                row.update({f"visible_{k}": v for k, v in visible_validation_summary(path, task).items() if k != "first_wrong"})
            except Exception as exc:
                row["visible_error"] = repr(exc)
        rows.append(row)
    zip_path = create_submission_zip(out_dir)
    return rows, zip_path


In [2]:
from pathlib import Path
import ast
import json
import pandas as pd

ROOT = Path.cwd()

FAMILY = 'fill_enclosed_regions'
SUBTYPE = 'nonlocal_1color'
MODEL_VERSION = 'fill-additive-nonlocal-1color-v0.68-task255-static-symbolic-tensor-builder-v3'
DATA_DIR, BASE_OUT_DIR = default_paths()
OUT_DIR = BASE_OUT_DIR / f'{FAMILY}_{SUBTYPE}_task255_static_symbolic_tensor_builder_v3_v68'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR =', DATA_DIR)
print('OUT_DIR =', OUT_DIR)
print('MODEL_VERSION =', MODEL_VERSION)


DATA_DIR = /kaggle/input/competitions/neurogolf-2026
OUT_DIR = /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color_task255_static_symbolic_tensor_builder_v3_v68
MODEL_VERSION = fill-additive-nonlocal-1color-v0.68-task255-static-symbolic-tensor-builder-v3


In [3]:
# ONNX dependency setup for model export.
# Dry-run rule fitting can run without ONNX, but build_family_submission
# must import onnx to create taskNNN.onnx files.
import importlib.util
import subprocess
import sys

missing = [pkg for pkg in ['onnx', 'onnxruntime'] if importlib.util.find_spec(pkg) is None]
if missing:
    print('Installing missing ONNX packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])


import onnx
import onnxruntime as ort
from onnx import TensorProto, helper, numpy_helper

print('onnx:', onnx.__version__)
print('onnxruntime:', ort.__version__)

Installing missing ONNX packages: ['onnxruntime']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 72.3 MB/s eta 0:00:00
onnx: 1.20.1
onnxruntime: 1.27.0


In [4]:
# update for each task
task_map = load_task_type_map()
family_df = task_map[task_map.primary_family == FAMILY].copy()


def parse_color_list(value):
    if pd.isna(value) or value == '':
        return []
    if isinstance(value, list):
        return value
    try:
        return list(ast.literal_eval(str(value)))
    except Exception:
        return []

family_df['parsed_new_output_colors'] = family_df['new_output_color_list'].apply(parse_color_list)
nonlocal_1color_df = family_df[
    family_df['candidate_flags'].fillna('').str.contains('adds_new_color_preserves_input')
    & ~family_df['candidate_flags'].fillna('').str.contains('local_3x3_consistent')
    & family_df['parsed_new_output_colors'].apply(lambda colors: len(colors) == 1)
].copy()
nonlocal_1color_df = nonlocal_1color_df.sort_values('task_id').reset_index(drop=True)
task_ids = ['task255']
nonlocal_1color_df = nonlocal_1color_df[nonlocal_1color_df['task_id'].isin(task_ids)].copy().reset_index(drop=True)

print('family:', FAMILY)
print('subtype:', SUBTYPE)
print('family tasks:', len(family_df))
print('selected nonlocal_1color task255 symbolic tensor v3 tasks:', len(task_ids))
print(task_ids)
display(nonlocal_1color_df.head(10))



family: fill_enclosed_regions
subtype: nonlocal_1color
family tasks: 59
selected nonlocal_1color task255 symbolic tensor v3 tasks: 1
['task255']


,task_id,task_num,primary_family,confidence,candidate_flags,n_train,n_test,n_arc_gen,n_examples,shape_relation,...,mapping_conflicts,fixed_geometric_transforms,local_3x3_score,local_3x3_conflicts,local_3x3_samples,input_nonzero_preserved_ratio,added_nonzero_cells,changed_cells,notes,parsed_new_output_colors
0,task255,255,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,3,1,261,265,same_shape_variable_size,...,61977,NaN,0.752111,13386,54000,1.0,61977,61977,Same shape; input is mostly preserved while ne...,[3]


In [5]:
# Inspect one scoped task quickly.
if task_ids:
    sample_task_id = task_ids[0]
    sample_task = load_task(DATA_DIR, sample_task_id)
    print(sample_task_id, 'examples:', len(all_examples(sample_task)))
    print('first input shape:', grid_shape(sample_task['train'][0]['input']))
    print('first output shape:', grid_shape(sample_task['train'][0]['output']))
    print('first input:', sample_task['train'][0]['input'])
    print('first output:', sample_task['train'][0]['output'])
else:
    print('No tasks currently mapped to this subtype.')

task255 examples: 265
first input shape: (30, 30)
first output shape: (30, 30)
first input: [[8, 8, 0, 8, 0, 8, 0, 8, 8, 8, 8, 8, 0, 8, 8, 8, 0, 8, 0, 0, 8, 0, 8, 0, 0, 0, 8, 8, 0, 8], [0, 0, 0, 8, 8, 8, 8, 0, 0, 8, 0, 8, 0, 0, 8, 8, 0, 0, 8, 0, 0, 0, 0, 0, 8, 8, 8, 8, 0, 8], [8, 0, 0, 0, 8, 8, 0, 0, 8, 0, 8, 8, 0, 8, 8, 0, 8, 0, 8, 0, 8, 8, 8, 8, 0, 0, 8, 0, 0, 0], [0, 8, 8, 0, 0, 0, 0, 8, 8, 0, 0, 0, 0, 8, 8, 0, 8, 8, 0, 0, 0, 8, 8, 0, 8, 0, 0, 0, 0, 0], [8, 8, 8, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 0, 0, 8, 0, 8, 8, 0, 0, 8], [0, 8, 0, 0, 0, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 0, 8, 0, 8, 0, 0, 0, 8], [0, 8, 8, 8, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 0, 0, 0, 0, 0, 8, 0, 8, 8, 8], [0, 8, 8, 8, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 8, 8, 8, 0, 0, 8, 8], [8, 0, 8, 8, 0, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 8, 0, 0, 8, 0, 8], [8, 8, 8, 0, 8, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [6]:
# nonlocal_1color selection table: these are the tasks this notebook version is responsible for.
selection_cols = [
    'task_id',
    'confidence',
    'n_train',
    'n_test',
    'n_arc_gen',
    'shape_relation',
    'input_shape_modes',
    'output_shape_modes',
    'input_color_list',
    'output_color_list',
    'new_output_color_list',
    'local_3x3_score',
    'local_3x3_conflicts',
    'added_nonzero_cells',
    'candidate_flags',
]
fill_selection = nonlocal_1color_df[selection_cols].reset_index(drop=True)
print('selected nonlocal_1color fill/additive tasks:', len(fill_selection))
display(fill_selection)

selected nonlocal_1color fill/additive tasks: 1


,task_id,confidence,n_train,n_test,n_arc_gen,shape_relation,input_shape_modes,output_shape_modes,input_color_list,output_color_list,new_output_color_list,local_3x3_score,local_3x3_conflicts,added_nonzero_cells,candidate_flags
0,task255,medium,3,1,261,same_shape_variable_size,30x30:265,30x30:265,"[0,1,2,4,5,6,7,8,9]","[0,1,2,3,4,5,6,7,8,9]",[3],0.752111,13386,61977,adds_new_color_preserves_input|new_output_colors


In [7]:
# task sepcific modelling modules

CLEAR = 10
ZERO_HOT = -1
NO_CHANGE = -2


def input_canvas(grid):
    canvas = np.full((H, W), CLEAR, dtype=np.int16)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if r < H and c < W:
                canvas[r, c] = int(color)
    return canvas


def output_canvas(grid):
    canvas = np.full((H, W), ZERO_HOT, dtype=np.int16)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if r < H and c < W:
                canvas[r, c] = int(color)
    return canvas


def task_color_sets(task):
    in_colors = sorted({int(v) for ex in all_examples(task) for row in ex['input'] for v in row})
    out_colors = sorted({int(v) for ex in all_examples(task) for row in ex['output'] for v in row})
    return in_colors, out_colors


def exact_grid_match(task, predictor):
    right = 0
    total = 0
    first_wrong = None
    for split in ['train', 'test', 'arc-gen']:
        for idx, ex in enumerate(task.get(split, [])):
            total += 1
            pred = predictor(ex['input'])
            if pred == ex['output']:
                right += 1
            elif first_wrong is None:
                first_wrong = (split, idx)
    return right, total, first_wrong



def evaluate_predictor_on_task(task, predictor):
    right = 0
    total = 0
    first_wrong = None
    split_counts = {}
    for split in ['train', 'test', 'arc-gen']:
        split_right = 0
        split_total = 0
        for idx, ex in enumerate(task.get(split, [])):
            split_total += 1
            total += 1
            pred = predictor(ex['input'])
            if pred == ex['output']:
                split_right += 1
                right += 1
            elif first_wrong is None:
                first_wrong = f'{split}[{idx}]'
        split_counts[f'{split}_right'] = split_right
        split_counts[f'{split}_total'] = split_total
    return {
        'right': right,
        'total': total,
        'accuracy': right / total if total else None,
        'first_wrong': first_wrong,
        **split_counts,
    }


# --- v68 task255 static symbolic tensor builder v3 override ---------------------
# Broad symbolic candidate family from geometry, exported as one ONNX graph.
# This replaces the v2 accepted-rectangle table route for task255.


def task255_symbolic_arr(x): return np.asarray(x, dtype=np.int64)

def make_general_rectangle_candidates(H=30, W=30, min_area=39, min_long=13, max_short=15):
    """Broad symbolic candidate family from grid geometry and train-inferred scale bounds.
    This is not an accepted-rectangle list and does not depend on visible outputs.
    It contains all border-touching rectangles satisfying broad scale constraints.
    """
    cands = []
    for r0 in range(H):
        for r1 in range(r0, H):
            h = r1 - r0 + 1
            for c0 in range(W):
                for c1 in range(c0, W):
                    w = c1 - c0 + 1
                    touches_border = (r0 == 0 or r1 == H - 1 or c0 == 0 or c1 == W - 1)
                    if touches_border and h * w >= min_area and max(h, w) >= min_long and min(h, w) <= max_short:
                        cands.append((r0, r1, c0, c1, h, w, h * w))
    return sorted(cands, key=lambda x: (-x[-1], x[0], x[2], x[1], x[3]))

TASK255_SYMBOLIC_TENSOR_CANDIDATES = make_general_rectangle_candidates(H, W)
print("task255 symbolic tensor candidate count", len(TASK255_SYMBOLIC_TENSOR_CANDIDATES))

def prefix_sum_bool(mask):
    return np.pad(mask.astype(np.int64), ((1,0),(1,0)), constant_values=0).cumsum(0).cumsum(1)

def rect_sum(ps, r0, r1, c0, c1):
    return int(ps[r1+1, c1+1] - ps[r0, c1+1] - ps[r1+1, c0] + ps[r0, c0])

def run_ge(mask, L=4, axis=1):
    H, W = mask.shape
    out = np.zeros_like(mask, dtype=bool)
    if axis == 1:
        cs = np.pad(mask.astype(np.int64), ((0,0),(1,0)), constant_values=0).cumsum(1)
        win = cs[:, L:] - cs[:, :-L]
        for c0 in range(W - L + 1):
            out[:, c0:c0+L] |= (win[:, c0:c0+1] == L)
    else:
        cs = np.pad(mask.astype(np.int64), ((1,0),(0,0)), constant_values=0).cumsum(0)
        win = cs[L:, :] - cs[:-L, :]
        for r0 in range(H - L + 1):
            out[r0:r0+L, :] |= (win[r0:r0+1, :] == L)
    return out

def predict_task255_symbolic_tensor_mirror(grid, iterations=5):
    """Python mirror of the ONNX graph: broad parallel rectangle evidence + greedy largest-area acceptance."""
    g = task255_symbolic_arr(grid); zero = (g == 0); ps = prefix_sum_bool(zero)
    mark = np.zeros((H, W), dtype=bool)
    valid_candidates = []
    for r0, r1, c0, c1, h, w, area in TASK255_SYMBOLIC_TENSOR_CANDIDATES:
        if rect_sum(ps, r0, r1, c0, c1) != area:
            continue
        can_ext = False
        if r0 > 0 and rect_sum(ps, r0-1, r0-1, c0, c1) == w: can_ext = True
        if r1 < H-1 and rect_sum(ps, r1+1, r1+1, c0, c1) == w: can_ext = True
        if c0 > 0 and rect_sum(ps, r0, r1, c0-1, c0-1) == h: can_ext = True
        if c1 < W-1 and rect_sum(ps, r0, r1, c1+1, c1+1) == h: can_ext = True
        if can_ext:
            continue
        rr0 = r0 + (1 if r0 > 0 else 0)
        rr1 = r1 - (1 if r1 < H-1 else 0)
        cc0 = c0 + (1 if c0 > 0 else 0)
        cc1 = c1 - (1 if c1 < W-1 else 0)
        if rr0 > rr1 or cc0 > cc1:
            continue
        cand = np.zeros((H, W), dtype=bool)
        cand[rr0:rr1+1, cc0:cc1+1] = True
        cand &= zero
        valid_candidates.append((area, cand))
    for _ in range(iterations):
        best = None
        for area, cand in valid_candidates:
            new = cand & (~mark)
            add = new & (run_ge(new, 4, axis=1) | run_ge(new, 4, axis=0))
            if int(add.sum()) >= 6 and (best is None or area > best[0]):
                best = (area, add)
        if best is None:
            break
        mark |= best[1]
    out = g.copy(); out[mark] = 3
    return out.tolist()


def make_task255_static_symbolic_tensor_builder_v3_model(payload=None):
    require_onnx()
    cands = TASK255_SYMBOLIC_TENSOR_CANDIDATES
    N = len(cands)
    r0 = np.array([x[0] for x in cands], np.float32)
    r1 = np.array([x[1] for x in cands], np.float32)
    c0 = np.array([x[2] for x in cands], np.float32)
    c1 = np.array([x[3] for x in cands], np.float32)
    hh = np.array([x[4] for x in cands], np.float32)
    ww = np.array([x[5] for x in cands], np.float32)
    area = np.array([x[6] for x in cands], np.float32)
    nodes, inits = [], []
    def init(n, a): inits.append(numpy_helper.from_array(np.asarray(a), name=n))
    def node(op, ins, outs, name=None, **attrs): nodes.append(helper.make_node(op, ins, outs, name=name, **attrs))
    init('idx0_i64', np.array(0, np.int64)); init('half_f', np.array(0.5, np.float32)); init('zero_f', np.array(0, np.float32)); init('one_f', np.array(1, np.float32)); init('four_f', np.array(4, np.float32)); init('six_f', np.array(6, np.float32)); init('neg_one_f', np.array(-1, np.float32)); init('twentynine_f', np.array(29, np.float32))
    for n, a in [('axes0',[0]), ('axes1',[1]), ('axes12',[1,2]), ('axes_unsq12',[1,2]), ('axes_unsq01',[0,1])]:
        init(n, np.array(a, np.int64))
    init('row_idx', np.arange(H, dtype=np.float32).reshape(1,H,1))
    init('col_idx', np.arange(W, dtype=np.float32).reshape(1,1,W))
    for nm, a in [('r0v',r0), ('r1v',r1), ('c0v',c0), ('c1v',c1), ('hv',hh), ('wv',ww), ('areav',area)]:
        init(nm, a)
    init('kh', np.ones((1,1,1,4), np.float32)); init('kv', np.ones((1,1,4,1), np.float32))
    basis = np.zeros((1,10,1,1), np.float32); basis[0,3,0,0] = 1; init('basis3', basis)
    init('init_mark', np.zeros((H,W), np.bool_))
    for base in ['r0','r1','c0','c1']:
        node('Unsqueeze', [base+'v','axes_unsq12'], [base+'b'], name='unsq_'+base)
    node('Gather', ['input','idx0_i64'], ['chan0_bhw'], axis=1, name='gather_zero_channel')
    node('Squeeze', ['chan0_bhw','axes0'], ['chan0'], name='squeeze_batch')
    node('Greater', ['chan0','half_f'], ['zero_mask'], name='zero_mask_bool')
    node('GreaterOrEqual', ['row_idx','r0b'], ['row_ge_r0']); node('LessOrEqual', ['row_idx','r1b'], ['row_le_r1']); node('And', ['row_ge_r0','row_le_r1'], ['row_rect'])
    node('GreaterOrEqual', ['col_idx','c0b'], ['col_ge_c0']); node('LessOrEqual', ['col_idx','c1b'], ['col_le_c1']); node('And', ['col_ge_c0','col_le_c1'], ['col_rect'])
    node('And', ['row_rect','col_rect'], ['rect_mask'])
    node('And', ['rect_mask','zero_mask'], ['zero_rect'])
    node('Cast', ['zero_rect'], ['zero_rect_f'], to=TensorProto.FLOAT)
    node('ReduceSum', ['zero_rect_f','axes12'], ['rect_sum'], keepdims=0)
    node('Equal', ['rect_sum','areav'], ['all_zero'])
    node('Greater', ['r0v','zero_f'], ['r0_gt0']); node('Less', ['r1v','twentynine_f'], ['r1_lt29']); node('Greater', ['c0v','zero_f'], ['c0_gt0']); node('Less', ['c1v','twentynine_f'], ['c1_lt29'])
    node('Sub', ['r0b','one_f'], ['r_up_b']); node('Add', ['r1b','one_f'], ['r_down_b']); node('Sub', ['c0b','one_f'], ['c_left_b']); node('Add', ['c1b','one_f'], ['c_right_b'])
    node('Equal', ['row_idx','r_up_b'], ['row_up']); node('And', ['row_up','col_rect'], ['mask_up']); node('And', ['mask_up','zero_mask'], ['zero_up']); node('Cast', ['zero_up'], ['zero_up_f'], to=TensorProto.FLOAT); node('ReduceSum', ['zero_up_f','axes12'], ['sum_up'], keepdims=0); node('Equal', ['sum_up','wv'], ['up_full']); node('And', ['r0_gt0','up_full'], ['can_up'])
    node('Equal', ['row_idx','r_down_b'], ['row_down']); node('And', ['row_down','col_rect'], ['mask_down']); node('And', ['mask_down','zero_mask'], ['zero_down']); node('Cast', ['zero_down'], ['zero_down_f'], to=TensorProto.FLOAT); node('ReduceSum', ['zero_down_f','axes12'], ['sum_down'], keepdims=0); node('Equal', ['sum_down','wv'], ['down_full']); node('And', ['r1_lt29','down_full'], ['can_down'])
    node('Equal', ['col_idx','c_left_b'], ['col_left']); node('And', ['row_rect','col_left'], ['mask_left']); node('And', ['mask_left','zero_mask'], ['zero_left']); node('Cast', ['zero_left'], ['zero_left_f'], to=TensorProto.FLOAT); node('ReduceSum', ['zero_left_f','axes12'], ['sum_left'], keepdims=0); node('Equal', ['sum_left','hv'], ['left_full']); node('And', ['c0_gt0','left_full'], ['can_left'])
    node('Equal', ['col_idx','c_right_b'], ['col_right']); node('And', ['row_rect','col_right'], ['mask_right']); node('And', ['mask_right','zero_mask'], ['zero_right']); node('Cast', ['zero_right'], ['zero_right_f'], to=TensorProto.FLOAT); node('ReduceSum', ['zero_right_f','axes12'], ['sum_right'], keepdims=0); node('Equal', ['sum_right','hv'], ['right_full']); node('And', ['c1_lt29','right_full'], ['can_right'])
    node('Or', ['can_up','can_down'], ['can_ud']); node('Or', ['can_left','can_right'], ['can_lr']); node('Or', ['can_ud','can_lr'], ['can_ext']); node('Not', ['can_ext'], ['maximal']); node('And', ['all_zero','maximal'], ['valid_base'])
    node('Cast', ['r0_gt0'], ['r0_gt0_f'], to=TensorProto.FLOAT); node('Cast', ['r1_lt29'], ['r1_lt29_f'], to=TensorProto.FLOAT); node('Cast', ['c0_gt0'], ['c0_gt0_f'], to=TensorProto.FLOAT); node('Cast', ['c1_lt29'], ['c1_lt29_f'], to=TensorProto.FLOAT)
    node('Add', ['r0v','r0_gt0_f'], ['rr0v']); node('Sub', ['r1v','r1_lt29_f'], ['rr1v']); node('Add', ['c0v','c0_gt0_f'], ['cc0v']); node('Sub', ['c1v','c1_lt29_f'], ['cc1v'])
    for base in ['rr0','rr1','cc0','cc1']:
        node('Unsqueeze', [base+'v','axes_unsq12'], [base+'b'], name='unsq_'+base)
    node('GreaterOrEqual', ['row_idx','rr0b'], ['row_ge_rr0']); node('LessOrEqual', ['row_idx','rr1b'], ['row_le_rr1']); node('And', ['row_ge_rr0','row_le_rr1'], ['row_er'])
    node('GreaterOrEqual', ['col_idx','cc0b'], ['col_ge_cc0']); node('LessOrEqual', ['col_idx','cc1b'], ['col_le_cc1']); node('And', ['col_ge_cc0','col_le_cc1'], ['col_er'])
    node('And', ['row_er','col_er'], ['er_mask']); node('And', ['er_mask','zero_mask'], ['cand_base'])
    mark = 'init_mark'
    for t in range(5):
        p = f'it{t}_'
        node('Not', [mark], [p+'not_mark']); node('And', ['cand_base',p+'not_mark'], [p+'new']); node('Cast', [p+'new'], [p+'new_f'], to=TensorProto.FLOAT); node('Unsqueeze', [p+'new_f','axes1'], [p+'new4'])
        node('Conv', [p+'new4','kh'], [p+'seg_h_sum'], name=p+'conv_h'); node('Equal', [p+'seg_h_sum','four_f'], [p+'seg_h_bool']); node('Cast', [p+'seg_h_bool'], [p+'seg_h_f'], to=TensorProto.FLOAT); node('ConvTranspose', [p+'seg_h_f','kh'], [p+'run_h_sum'], name=p+'convt_h'); node('Greater', [p+'run_h_sum','zero_f'], [p+'run_h4']); node('Squeeze', [p+'run_h4','axes1'], [p+'run_h'])
        node('Conv', [p+'new4','kv'], [p+'seg_v_sum'], name=p+'conv_v'); node('Equal', [p+'seg_v_sum','four_f'], [p+'seg_v_bool']); node('Cast', [p+'seg_v_bool'], [p+'seg_v_f'], to=TensorProto.FLOAT); node('ConvTranspose', [p+'seg_v_f','kv'], [p+'run_v_sum'], name=p+'convt_v'); node('Greater', [p+'run_v_sum','zero_f'], [p+'run_v4']); node('Squeeze', [p+'run_v4','axes1'], [p+'run_v'])
        node('Or', [p+'run_h',p+'run_v'], [p+'run4']); node('And', [p+'new',p+'run4'], [p+'add_mask_all']); node('Cast', [p+'add_mask_all'], [p+'add_mask_all_f'], to=TensorProto.FLOAT); node('ReduceSum', [p+'add_mask_all_f','axes12'], [p+'add_count'], keepdims=0)
        node('GreaterOrEqual', [p+'add_count','six_f'], [p+'enough']); node('And', ['valid_base',p+'enough'], [p+'accept']); node('Where', [p+'accept','areav','neg_one_f'], [p+'scores']); node('ArgMax', [p+'scores'], [p+'best_idx'], axis=0, keepdims=0, select_last_index=0); node('ReduceMax', [p+'scores'], [p+'best_score'], keepdims=0); node('Greater', [p+'best_score','zero_f'], [p+'has_candidate'])
        node('Gather', [p+'add_mask_all',p+'best_idx'], [p+'selected_add'], axis=0); node('And', [p+'selected_add',p+'has_candidate'], [p+'selected_add_ok']); node('Or', [mark,p+'selected_add_ok'], [p+'mark_out']); mark = p+'mark_out'
    node('Cast', [mark], ['mark_f'], to=TensorProto.FLOAT); node('Unsqueeze', ['mark_f','axes_unsq01'], ['mark4']); node('Sub', ['one_f','mark4'], ['not_mark4']); node('Mul', ['input','not_mark4'], ['preserve']); node('Mul', ['basis3','mark4'], ['added3']); node('Add', ['preserve','added3'], ['output'])
    graph = helper.make_graph(nodes, 'task255_static_symbolic_tensor_builder_v3', [helper.make_tensor_value_info('input', TensorProto.FLOAT, [1,10,H,W])], [helper.make_tensor_value_info('output', TensorProto.FLOAT, [1,10,H,W])], initializer=inits)
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 18)], ir_version=10)
    onnx.checker.check_model(model)
    onnx.checker.check_model(model)
    return model


def fit_task255_static_symbolic_tensor_builder_v3(task):
    right, total, first_wrong = exact_grid_match(task, predict_task255_symbolic_tensor_mirror)
    return {'kind': 'task255_static_symbolic_tensor_builder_v3'}, {
        'ok': right == total,
        'trainer': 'task255_static_symbolic_tensor_builder_v3',
        'model_version': MODEL_VERSION,
        'semantic_kind': 'broad_border_zero_rectangle_symbolic_tensor_search_with_run4_filter',
        'visible_right': right,
        'visible_total': total,
        'first_wrong': first_wrong,
        'candidate_rectangles': len(TASK255_SYMBOLIC_TENSOR_CANDIDATES),
        'estimated_model_size_bytes': 794352,
        'generalization_note': 'broad symbolic geometry family; not an accepted-rectangle lookup table',
    }


EXPORTABLE_PREDICTORS = {'task255': ('task255_static_symbolic_tensor_builder_v3', predict_task255_symbolic_tensor_mirror)}
SIMULATOR_ONLY_PREDICTORS = {}


def train_family_task(task):
    payload, info = fit_task255_static_symbolic_tensor_builder_v3(task)
    if info.get('ok'):
        return make_task255_static_symbolic_tensor_builder_v3_model(payload), info
    return None, {
        'ok': False,
        'trainer': 'identity_fallback_export',
        'model_version': MODEL_VERSION,
        'reason': info.get('reason', 'task255-only notebook: symbolic tensor v3 simulation failed'),
    }

task255 symbolic tensor candidate count 27914


In [8]:
# Correctness matrix for task255-only static symbolic tensor builder v3 run.
# The target is to move every task from no_predictor/identity to visible_correct with a semantic predictor.
if 'task_ids' not in globals():
    task_ids = ['task255']

correctness_rows = []
for task_id in task_ids:
    task = load_task(DATA_DIR, task_id)
    status = 'no_predictor'
    rule_name = None
    export_status = 'none'
    predictor = None

    if task_id in EXPORTABLE_PREDICTORS:
        rule_name, predictor = EXPORTABLE_PREDICTORS[task_id]
        export_status = 'exportable'
        status = 'has_predictor'
    elif task_id in SIMULATOR_ONLY_PREDICTORS:
        rule_name, predictor = SIMULATOR_ONLY_PREDICTORS[task_id]
        export_status = 'simulator_only'
        status = 'has_predictor'
    else:
        rule_name = 'identity_baseline'
        predictor = identity_grid_predictor
        export_status = 'identity_fallback'
        status = 'identity_baseline'

    summary = evaluate_predictor_on_task(task, predictor)
    if status == 'has_predictor':
        status = 'visible_correct' if summary['right'] == summary['total'] else 'partial'
    elif status == 'identity_baseline' and summary['right'] == summary['total']:
        status = 'identity_correct'

    correctness_rows.append({
        'task_id': task_id,
        'status': status,
        'rule_name': rule_name,
        'export_status': export_status,
        'right': summary['right'],
        'total': summary['total'],
        'accuracy': summary['accuracy'],
        'first_wrong': summary['first_wrong'],
        'train_right': summary['train_right'],
        'train_total': summary['train_total'],
        'test_right': summary['test_right'],
        'test_total': summary['test_total'],
        'arc_gen_right': summary['arc-gen_right'],
        'arc_gen_total': summary['arc-gen_total'],
    })

correctness_df = pd.DataFrame(correctness_rows)
display(correctness_df)
display(correctness_df['status'].value_counts().rename_axis('status').reset_index(name='count'))
display(correctness_df['export_status'].value_counts().rename_axis('export_status').reset_index(name='count'))

unsolved_df = correctness_df[~correctness_df['status'].isin(['visible_correct', 'identity_correct'])].copy()
print('visible-correct tasks:', int((correctness_df['status'] == 'visible_correct').sum()), '/', len(correctness_df))
print('unsolved tasks:', len(unsolved_df))
display(unsolved_df[['task_id', 'status', 'right', 'total', 'accuracy', 'first_wrong']])


,task_id,status,rule_name,export_status,right,total,accuracy,first_wrong,train_right,train_total,test_right,test_total,arc_gen_right,arc_gen_total
0,task255,visible_correct,task255_static_symbolic_tensor_builder_v3,exportable,265,265,1.0,None,3,3,1,1,261,261


,status,count
0,visible_correct,1


,export_status,count
0,exportable,1


visible-correct tasks: 1 / 1
unsolved tasks: 0


,task_id,status,right,total,accuracy,first_wrong


In [9]:
# Build one model file for task255-only static symbolic tensor builder v3 run.
# The build uses exportable semantic models where available and identity fallback otherwise.
import shutil

if 'task_ids' not in globals():
    task_ids = ['task255']

# Clear stale models from earlier runs before creating this scoped zip.
for old_model_path in OUT_DIR.glob('task*.onnx'):
    old_model_path.unlink()

print('pre-build task ids:', task_ids)
rows, zip_path = build_family_submission(
    FAMILY,
    train_family_task,
    DATA_DIR,
    OUT_DIR,
    fallback_identity=True,
    validate=False,
    task_ids_override=task_ids,
)

result_df = pd.DataFrame(rows)
display(result_df)
saved_count = int(result_df.get('saved', pd.Series(dtype=bool)).sum()) if len(result_df) else 0
print('selected task255-only tasks:', len(task_ids))
print('models saved:', saved_count)
if len(result_df) and 'trainer' in result_df:
    display(result_df['trainer'].fillna('none').value_counts().rename_axis('trainer').reset_index(name='count'))

expected_names = {f'{task_id}.onnx' for task_id in task_ids}
actual_names = {path.name for path in OUT_DIR.glob('task*.onnx')}
print('missing models:', sorted(expected_names - actual_names))
print('extra models:', sorted(actual_names - expected_names))
assert not (expected_names - actual_names), 'missing scoped task models'
assert not (actual_names - expected_names), 'found stale or out-of-scope task models'

# Kaggle looks for /kaggle/working/submission.zip when submitting from a notebook.
submission_zip = Path('/kaggle/working/submission.zip') if Path('/kaggle/working').exists() else Path.cwd() / 'submission.zip'
shutil.copy2(zip_path, submission_zip)
print('family zip:', zip_path)
print('kaggle submission zip:', submission_zip)


pre-build task ids: ['task255']


,task_id,saved,path,ok,trainer,model_version,semantic_kind,visible_right,visible_total,first_wrong,candidate_rectangles,estimated_model_size_bytes,generalization_note
0,task255,True,/kaggle/working/working_submission/fill_enclos...,True,task255_static_symbolic_tensor_builder_v3,fill-additive-nonlocal-1color-v0.68-task255-st...,broad_border_zero_rectangle_symbolic_tensor_se...,265,265,None,27914,794352,broad symbolic geometry family; not an accepte...


selected task255-only tasks: 1
models saved: 1


,trainer,count
0,task255_static_symbolic_tensor_builder_v3,1


missing models: []
extra models: []
family zip: /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color_task255_static_symbolic_tensor_builder_v3_v68/submission.zip
kaggle submission zip: /kaggle/working/submission.zip


In [10]:
# Model/version manifest for this notebook run.
run_manifest = {
    'family': FAMILY,
    'subtype': SUBTYPE,
    'model_version': MODEL_VERSION,
    'task_count': len(task_ids),
    'task_ids': task_ids,
    'out_dir': str(OUT_DIR),
    'strategy': 'task255-only static symbolic tensor ONNX builder v3 over broad border-zero rectangle candidates',
    'exportable_predictors': sorted(EXPORTABLE_PREDICTORS.keys()) if 'EXPORTABLE_PREDICTORS' in globals() else [],
    'simulator_only_predictors': sorted(SIMULATOR_ONLY_PREDICTORS.keys()) if 'SIMULATOR_ONLY_PREDICTORS' in globals() else [],
}
run_manifest


{'family': 'fill_enclosed_regions',
 'subtype': 'nonlocal_1color',
 'model_version': 'fill-additive-nonlocal-1color-v0.68-task255-static-symbolic-tensor-builder-v3',
 'task_count': 1,
 'task_ids': ['task255'],
 'out_dir': '/kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color_task255_static_symbolic_tensor_builder_v3_v68',
 'strategy': 'task255-only static symbolic tensor ONNX builder v3 over broad border-zero rectangle candidates',
 'exportable_predictors': ['task255'],
 'simulator_only_predictors': []}

In [11]:
# Optional: validate saved ONNX models on visible examples.
# This can be slow for large families and requires onnxruntime.
# 'path' in rows: '/kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color_task255_static_onnx_builder_v2_v67/task255.onnx'


validate_rows = []

for row in rows:
    if not row.get('saved'):
        continue
    task = load_task(DATA_DIR, row['task_id'])
    summary = visible_validation_summary(row['path'], task)
    validate_rows.append({
        'task_id': row['task_id'],
        'right': summary['right'],
        'wrong': summary['wrong'],
    })

print(pd.DataFrame(validate_rows))
print("input and expected output: ", summary['first_wrong_expected'])
print("actual model output: ", summary['first_wrong_actual'])

len of examples:  265
example 1
example 2
example 3
example 4
example 5
example 6
example 7
example 8
example 9
example 10
example 11
example 12
example 13
example 14
example 15
example 16
example 17
example 18
example 19
example 20
example 21
example 22
example 23
example 24
example 25
example 26
example 27
example 28
example 29
example 30
example 31
example 32
example 33
example 34
example 35
example 36
example 37
example 38
example 39
example 40
example 41
example 42
example 43
example 44
example 45
example 46
example 47
example 48
example 49
example 50
example 51
example 52
example 53
example 54
example 55
example 56
example 57
example 58
example 59
example 60
example 61
example 62
example 63
example 64
example 65
example 66
example 67
example 68
example 69
example 70
example 71
example 72
example 73
example 74
example 75
example 76
example 77
example 78
example 79
example 80
example 81
example 82
example 83
example 84
example 85
example 86
example 87
example 88
example 89
example 

In [12]:
# Architecture, performance, and memory report for saved models.
# This cell expects train_family_task to save one or more ONNX models.
# It reports the metrics the competition cares about: file size, parameter
# count, and memory profile, plus train/test/arc-gen exact-match performance.

report_rows = []
for row in rows:
    if not row.get('saved'):
        continue
    task = load_task(DATA_DIR, row['task_id'])
    try:
        report = model_report(row['path'], task=task)
        arch = report['architecture']
        mem = report['memory_profile']
        perf = report['performance']
        report_rows.append({
            'task_id': row['task_id'],
            'model_version': MODEL_VERSION,
            'file_size_bytes': arch.get('file_size_bytes'),
            'params': arch.get('params'),
            'nodes': arch.get('nodes'),
            'op_counts': json.dumps(arch.get('op_counts', {}), sort_keys=True),
            'static_memory_bytes': mem.get('static_memory_bytes'),
            'runtime_memory_bytes': mem.get('runtime_memory_bytes'),
            'train_right': perf['train']['right'],
            'train_total': perf['train']['total'],
            'train_accuracy': perf['train']['accuracy'],
            'test_right': perf['test']['right'],
            'test_total': perf['test']['total'],
            'test_accuracy': perf['test']['accuracy'],
            'arc_gen_right': perf['arc_gen']['right'],
            'arc_gen_total': perf['arc_gen']['total'],
            'arc_gen_accuracy': perf['arc_gen']['accuracy'],
            'visible_right': perf['visible_all']['right'],
            'visible_total': perf['visible_all']['total'],
            'visible_accuracy': perf['visible_all']['accuracy'],
        })
    except Exception as exc:
        report_rows.append({
            'task_id': row['task_id'],
            'model_version': MODEL_VERSION,
            'profile_error': repr(exc),
        })

profile_df = pd.DataFrame(report_rows)
display(profile_df)

,task_id,model_version,file_size_bytes,params,nodes,op_counts,static_memory_bytes,runtime_memory_bytes,train_right,train_total,train_accuracy,test_right,test_total,test_accuracy,arc_gen_right,arc_gen_total,arc_gen_accuracy,visible_right,visible_total,visible_accuracy
0,task255,fill-additive-nonlocal-1color-v0.68-task255-st...,794352,196392,230,"{""Add"": 5, ""And"": 41, ""ArgMax"": 5, ""Cast"": 30,...",5255898204,5255916204,3,3,1.0,1,1,1.0,261,261,1.0,265,265,1.0


In [13]:
# Persist run metadata next to the generated models.
if 'profile_df' in globals() and len(profile_df):
    profile_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_profile.csv'
    profile_df.to_csv(profile_path, index=False)
    print('wrote profile:', profile_path)

manifest_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(run_manifest, f, indent=2)
print('wrote manifest:', manifest_path)

wrote profile: /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color_task255_static_symbolic_tensor_builder_v3_v68/fill_enclosed_regions_fill-additive-nonlocal-1color-v0.68-task255-static-symbolic-tensor-builder-v3_profile.csv
wrote manifest: /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color_task255_static_symbolic_tensor_builder_v3_v68/fill_enclosed_regions_fill-additive-nonlocal-1color-v0.68-task255-static-symbolic-tensor-builder-v3_manifest.json
